# 02 · Features, alvo e divisão temporal

- **Alvo:** variação da cota em *h* horas (`cota[t+h] − cota[t]`), para h = 6, 12 e 24. A previsão absoluta é `cota[t] + variação`.
  Isso ajuda o modelo a lidar com cheias maiores do que as do treino (modelos não extrapolam bem níveis absolutos nunca vistos).
- **Features (tabular):** cota atual e variações (1, 3, 6, 12, 24 h) de todas as estações; chuva acumulada (6–48 h).
- **Sequências (GRU):** as últimas 48 h de todas as variáveis.
- **Divisão:** sempre por tempo (nunca sorteio). Treino → validação → teste (eventos de 2023–2024), com uma folga de 24 h entre os blocos.

In [ ]:
import os, sys
sys.path.append("..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, data, features, metrics, pipeline
plt.rcParams["figure.figsize"] = (11, 3.4)
plt.rcParams["axes.grid"] = True

In [ ]:
panel = pipeline.load_panel()
print(panel.shape, "| colunas:", list(panel.columns))

## Tabular: tamanhos e período de cada conjunto

In [ ]:
for alvo in config.TARGETS:
    if f"cota_{alvo}" not in panel:
        print("sem dados da estação-alvo", alvo); continue
    P = features.build_tabular(panel, alvo)
    linhas = []
    for nome, m in P.masks.items():
        if m.sum() == 0:
            linhas.append({"conjunto": nome, "amostras": 0}); continue
        linhas.append({"conjunto": nome, "amostras": int(m.sum()), "inicio": P.index[m][0], "fim": P.index[m][-1],
                       "cota_max_cm": float(P.base[m].max()), "maior_alta_24h_cm": float(P.Y[m][:, -1].max())})
    print(f"\n=== {config.STATIONS[alvo]['nome']} ({alvo}) | {P.X.shape[1]} features | horizontes {P.horizons} ===")
    display(pd.DataFrame(linhas).set_index("conjunto"))
    tr, te = P.base[P.masks["train"]].max(), P.base[P.masks["test"]].max() if P.masks["test"].any() else np.nan
    if te > tr:
        print(f"ATENÇÃO: a cota máxima no teste ({te:.0f} cm) passa a do treino ({tr:.0f} cm). "
              "O modelo precisará extrapolar; isso é uma limitação a discutir no artigo.")

## Sequências (para a GRU)

In [ ]:
alvo = config.TARGETS[0]
S = features.build_sequences(panel, alvo)
print("X:", S.X.shape, "(amostras, horas de histórico, variáveis) | Y:", S.Y.shape)
print("variáveis:", S.feature_names)
print({k: int(v.sum()) for k, v in S.masks.items()})

## Linha do tempo da divisão

In [ ]:
P = features.build_tabular(panel, alvo)
fig, ax = plt.subplots(figsize=(12, 3.6))
ax.plot(panel.index, panel[f"cota_{alvo}"], lw=0.5, color="0.6", label="cota")
for nome, cor in [("train", "tab:blue"), ("val", "tab:orange"), ("test", "tab:red")]:
    m = P.masks[nome]
    ax.plot(P.index[m], P.base[m], ".", ms=1.2, color=cor, label=nome)
for n, (i, f) in config.EVENTS.items():
    ax.axvspan(pd.Timestamp(i), pd.Timestamp(f), color="red", alpha=0.12)
ax.set_title(f"Divisão temporal — {config.STATIONS[alvo]['nome']}"); ax.legend(markerscale=6)
plt.show()